# Cross-Partner Threat Spread — Data Understanding

CRISP-DM phase 2. Scope locked in [README.md](README.md) §1.

**Goal:** join observations + NOI + PRISM; QC coverage; run first EDA checks before any new model.

**First EDA checks (from BU)**
1. Eligible-edge counts by partner / day
2. NOI vs actual spread (top-10 hit rate / AUC on eligible edges)
3. Missing-NOI rate on eligible edges
4. Days 8–14 positive rate
5. NOI as-of-T join audit (forecast stamp == T)

Outputs: `htoc_ml/analysis/_outputs/next_observed_relationships/data_understanding/`


## 0. Setup


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

import sys

from htoc.core.bootstrap import ensure_htoc_on_path

ensure_htoc_on_path()

from htoc.core.day import to_timestamp
from htoc.core.observations import ObservationData
from htoc.noi.config import ForecastConfig
from htoc.noi.feed_health import FeedHealth

REPO = Path(r"H:\HTOC")
ANALYSIS_NOI = REPO / "htoc_ml" / "analysis" / "noi"
if str(ANALYSIS_NOI) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_NOI))

import partner_relationships as nor  # noqa: E402

OUT = REPO / "htoc_ml" / "analysis" / "_outputs" / "next_observed_relationships" / "data_understanding"
OUT.mkdir(parents=True, exist_ok=True)

# Shorter window for first DU pass (full history later if needed)
TRAIN_DAYS = 90

cfg = nor.AnalysisConfig.production(REPO)
cfg = nor.AnalysisConfig(
    htoc_share_root=cfg.htoc_share_root,
    noi_save_dir=cfg.noi_save_dir,
    prism_xlsx=cfg.prism_xlsx,
    out_dir=OUT,
    train_days=TRAIN_DAYS,
    lookback_days=cfg.lookback_days,
)

print("share:", cfg.htoc_share_root)
print("noi:", cfg.noi_save_dir)
print("prism:", cfg.prism_xlsx)
print("out:", OUT)


## 1. Source inventory (observations, NOI, PRISM)


In [ ]:
forecast_config = ForecastConfig(
    htoc_share_root=cfg.htoc_share_root,
    save_dir=str(cfg.noi_save_dir),
    train_days=cfg.train_days,
    lookback_days=cfg.lookback_days,
    run_eval=False,
)

observations = ObservationData.load(
    obs_template=forecast_config.obs_template,
    train_days=cfg.train_days,
)
health = FeedHealth.from_data(observations.frame, today=observations.end_date)
prism = nor.load_prism_scores(cfg.prism_xlsx)

obs_desc = observations.describe()
print(obs_desc)
print("opdivs:", observations.labels.opdivs())
print("day range:", observations.day_min, "->", observations.day_max, "|", observations.end_date)

sev_counts = prism["severity"].value_counts()
medium_plus = prism.loc[prism["severity"].isin(nor.SEVERITY_KEEP)]
print("\nPRISM severity counts:")
display(sev_counts)
print(f"medium+ indicators: {len(medium_plus):,}")

(OUT / "obs_describe.txt").write_text(str(obs_desc), encoding="utf-8")
sev_counts.to_csv(OUT / "prism_severity_counts.csv")
medium_plus[["indicator", "severity", "prism_score"]].to_csv(OUT / "prism_medium_plus.csv", index=False)


## 2. NOI as-of-T join audit

For each analysis cutoff, confirm a matching `{Partner}_output_{YYYYMMDD}.csv` exists (stamp == T).


In [ ]:
cutoffs = nor.cutoff_days(observations.day_min, observations.day_max)
needed_dates = {to_timestamp(c).strftime("%Y%m%d") for c in cutoffs}
noi_cache = nor.preload_noi_forecasts(cfg.noi_save_dir, needed_dates)

audit_rows = []
for c in cutoffs:
    stamp = to_timestamp(c).strftime("%Y%m%d")
    frame = noi_cache.get(stamp, pd.DataFrame())
    n_partners = frame["Partner"].nunique() if not frame.empty and "Partner" in frame.columns else 0
    audit_rows.append(
        {
            "cutoff_day": c,
            "cutoff_stamp": stamp,
            "noi_rows": len(frame),
            "noi_partners": n_partners,
            "noi_file_present": len(frame) > 0,
        }
    )

noi_audit = pd.DataFrame(audit_rows)
print(
    f"cutoffs with NOI: {noi_audit['noi_file_present'].sum()} / {len(noi_audit)} "
    f"({100 * noi_audit['noi_file_present'].mean():.1f}%)"
)
display(noi_audit.tail(10))
noi_audit.to_csv(OUT / "noi_asof_t_audit.csv", index=False)


## 3. Eligible-edge table (medium+, A saw / B has not)

Uses `partner_relationships.build_event_table` (source active in last 7d, target not seen on cutoff).
Labels: `spread_7d` (days 1–7). Days 8–14 added in the next section.


In [ ]:
events = nor.build_event_table(
    observations,
    prism,
    health,
    forecast_config,
    noi_save_dir=cfg.noi_save_dir,
    medium_plus_only=True,
)
print(f"eligible-edge events: {len(events):,}")
if events.empty:
    raise RuntimeError("No events — check share paths / PRISM join / NOI coverage")

display(events.head())
print(events[["spread_7d", "noi_prob_7"]].describe())
events.to_csv(OUT / "eligible_edges_sample_window.csv", index=False)


## 4. First EDA checks


In [ ]:
# 4a. Eligible-edge counts by target partner / day
by_partner_day = (
    events.groupby(["cutoff_date", "target_partner"], observed=True)
    .size()
    .rename("n_edges")
    .reset_index()
)
by_partner = (
    events.groupby("target_partner", observed=True)
    .agg(
        n_edges=("indicator", "size"),
        n_indicators=("indicator", "nunique"),
        spread_7d_rate=("spread_7d", "mean"),
        missing_noi_rate=("noi_prob_7", lambda s: float(s.isna().mean())),
        mean_noi=("noi_prob_7", "mean"),
    )
    .sort_values("n_edges", ascending=False)
)
print("Edges by target partner:")
display(by_partner)
by_partner.to_csv(OUT / "eda_edges_by_target_partner.csv")
by_partner_day.to_csv(OUT / "eda_edges_by_partner_day.csv", index=False)

# 4b. Missing NOI on eligible edges
missing_noi = float(events["noi_prob_7"].isna().mean())
print(f"\nMissing NOI rate on eligible edges: {missing_noi:.1%}")

# 4c. Spread rate days 1-7
spread_7 = float(events["spread_7d"].mean())
print(f"Positive rate spread_7d (days 1-7): {spread_7:.2%}")


In [ ]:
# 4d. NOI vs actual spread (ranking quality on eligible edges)
scored = events.dropna(subset=["noi_prob_7"]).copy()
y = scored["spread_7d"].astype(int).to_numpy()
p = scored["noi_prob_7"].astype(float).to_numpy()

metrics = {
    "n_scored": len(scored),
    "spread_7d_rate": float(y.mean()) if len(y) else np.nan,
    "missing_noi_rate": missing_noi,
}
if y.sum() > 0 and (1 - y).sum() > 0:
    metrics["roc_auc"] = float(roc_auc_score(y, p))
    metrics["avg_precision"] = float(average_precision_score(y, p))
else:
    metrics["roc_auc"] = np.nan
    metrics["avg_precision"] = np.nan

# Top-10 hit rate by target partner (mean over cutoffs)
top10_rows = []
for (cutoff, partner), g in scored.groupby(["cutoff_day", "target_partner"], observed=True):
    top = g.nlargest(10, "noi_prob_7")
    top10_rows.append(
        {
            "cutoff_day": cutoff,
            "target_partner": partner,
            "top10_hit_rate": float(top["spread_7d"].mean()) if len(top) else np.nan,
            "base_rate": float(g["spread_7d"].mean()),
            "n": len(g),
        }
    )
top10 = pd.DataFrame(top10_rows)
metrics["top10_hit_rate_mean"] = float(top10["top10_hit_rate"].mean()) if len(top10) else np.nan
metrics["base_rate_mean"] = float(top10["base_rate"].mean()) if len(top10) else np.nan

metrics_df = pd.DataFrame([metrics])
print("NOI ranking quality on eligible edges:")
display(metrics_df)
display(top10.groupby("target_partner", observed=True)[["top10_hit_rate", "base_rate", "n"]].mean().sort_values("top10_hit_rate", ascending=False))

metrics_df.to_csv(OUT / "eda_noi_vs_spread_metrics.csv", index=False)
top10.to_csv(OUT / "eda_top10_by_partner_day.csv", index=False)


In [ ]:
# 4e. Days 8-14 positive rate (non-overlapping: first see in 8-14, not in 1-7)
# Recompute from observation labels for a sample of events (full join is heavy).

labels = observations.labels.as_dict()
empty = np.array([], dtype=int)


def spread_windows(target: str, indicator: str, cutoff_day: int) -> tuple[int, int]:
    dates = labels.get((target, indicator), empty)
    s7 = int(observations.labels.seen_next(dates, cutoff_day, 7))
    s14 = int(observations.labels.seen_next(dates, cutoff_day, 14))
    s8_14 = int(s14 == 1 and s7 == 0)
    return s7, s8_14


# Sample up to 50k rows for speed if needed
sample = events
if len(sample) > 50_000:
    sample = sample.sample(50_000, random_state=42)

s7_list, s814_list = [], []
for row in sample.itertuples(index=False):
    s7, s814 = spread_windows(row.target_partner, row.indicator, int(row.cutoff_day))
    s7_list.append(s7)
    s814_list.append(s814)

sample = sample.assign(spread_7d_check=s7_list, spread_8_14=s814_list)
rate_814 = float(sample["spread_8_14"].mean())
rate_7_check = float(sample["spread_7d_check"].mean())
print(f"Sample n={len(sample):,}")
print(f"spread_7d (recheck): {rate_7_check:.2%}")
print(f"spread_8_14 (non-overlapping): {rate_814:.2%}")

by_p_814 = (
    sample.groupby("target_partner", observed=True)["spread_8_14"]
    .mean()
    .rename("spread_8_14_rate")
    .sort_values(ascending=False)
)
display(by_p_814.to_frame())
by_p_814.to_csv(OUT / "eda_spread_8_14_by_partner.csv")
pd.DataFrame(
    [{"n": len(sample), "spread_7d_rate": rate_7_check, "spread_8_14_rate": rate_814}]
).to_csv(OUT / "eda_horizon_positive_rates.csv", index=False)


## 5. DU notes (pass 1 — 90-day window, 2026-06-19 → 2026-09-17)

- **Inventory:** 362k obs rows · 16k (opdiv,indicator) · 10 OpDivs · ~2,018 PRISM medium+
- **NOI as-of coverage:** **6 / 16** cutoffs had matching NOI files (37.5%) — large gap; need to map available NOI dates vs cutoff grid / CUTOFF_STEP
- **Eligibility volume:** **127,128** eligible edges; DHA/CDC largest target volumes; all partners represented
- **Missing NOI:** **90.5%** of eligible edges lack `noi_prob_7` — cold-target / missing-file risk is real; graph must handle missing P
- **NOI ranking vs spread (where NOI exists, n=12,051):** ROC-AUC **0.87** · AP **0.72** · Top-10 hit **75%** vs base **32%** — strong when present, but scored subset is enriched (29.7% positives vs 8.7% overall)
- **8–14 sparsity:** ~**2.7%** positive (non-overlapping) vs ~**8.8%** for 1–7 on sample — sparse but usable; keep for now
- **Open questions / BU revisit:**
  1. How to score edges with missing NOI (exclude from Top 10? impute? force model?)
  2. Align cutoff grid to days that actually have NOI files
  3. Confirm whether missing NOI ≈ “B never had X” (expected) vs file gaps
